In [1]:
import pandas as pd
import numpy as np

import re

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (

    accuracy_score,

    classification_report
)

import joblib

In [4]:
df = pd.read_csv(r"C:\Users\Personal\Downloads\rideflow_datasets.csv")

print(df.head())

      ride_id         timestamp pickup_zone  drop_zone  pickup_lat  \
0   95.247911  02-01-2025 01:30  Anna Nagar      Adyar   12.880239   
1  439.187633  05-01-2025 12:45     T Nagar   Tambaram   13.092441   
2  876.685389  09-01-2025 23:00  Anna Nagar   Tambaram   12.817965   
3  275.337197  03-01-2025 19:30     T Nagar  Velachery   13.125103   
4  106.743950  02-01-2025 02:30    Tambaram   Tambaram   13.143513   

   pickup_long   drop_lat  drop_long    driver_id  customer_id  ...  \
0    80.148410  13.028939  80.163941  1842.701958  6072.494896  ...   
1    80.165458  13.142711  80.149376  1186.296422  5942.228896  ...   
2    80.161839  12.943527  80.166040  1297.199801  5829.181415  ...   
3    80.143306  13.209127  80.126008  1765.474261  5429.619496  ...   
4    80.302596  13.078330  80.189672  1565.653849  5079.081677  ...   

   surge_multiplier  driver_rating  customer_rating  estimated_eta_min  \
0          1.001779       4.350624         4.037232          11.778023   
1   

In [ ]:
#CHECK FEEDBACK TEXT

In [5]:
print(df["feedback_text"].head())

0            Driver was polite
1    Driver cancelled suddenly
2    Driver cancelled suddenly
3              Good experience
4        Vehicle was not clean
Name: feedback_text, dtype: str


In [ ]:
#CREATE REALISTIC NLP TEXTS
#POSITIVE TEXTS

In [25]:
positive_reviews = [

    "Driver was very polite and friendly",

    "Amazing ride experience",

    "Ride was smooth and comfortable",

    "Driver arrived on time",

    "Very happy with the service",

    "Excellent driving and behavior",

    "Quick pickup and safe ride"
]

In [ ]:
#NEGATIVE TEXTS

In [26]:
negative_reviews = [

    "Driver was rude and late",

    "Very bad ride experience",

    "Ride got cancelled unexpectedly",

    "Driver did not arrive on time",

    "Terrible service and behavior",

    "Vehicle was not clean",

    "Very disappointed with the ride"
]

In [ ]:
#NEUTRAL TEXTS

In [27]:
neutral_reviews = [

    "Ride completed successfully",

    "Reached destination",

    "Trip completed",

    "Average experience",

    "Ride was okay",

    "Normal driving experience",

    "Reached on time"
]

In [ ]:
#GENERATE REALISTIC SENTIMENT DATA

In [28]:
import random

def generate_review(sentiment):

    if sentiment == "Positive":
        return random.choice(positive_reviews)

    elif sentiment == "Negative":
        return random.choice(negative_reviews)

    else:
        return random.choice(neutral_reviews)

In [ ]:
#CREATE SENTIMENT LABELS

In [29]:
def create_sentiment(row):

    if row["ride_status"] == "cancelled":

        return "Negative"

    elif row["driver_rating"] >= 4:

        return "Positive"

    else:

        return "Neutral"

In [ ]:
#APPLY LABELS

In [30]:
df["sentiment"] = df.apply(
    create_sentiment,
    axis=1
)

In [ ]:
#CREATE NEW FEEDBACK TEXT

In [31]:
df["feedback_text"] = df["sentiment"].apply(
    generate_review
)

In [ ]:
#CHECK RESULTS

In [32]:
print(
    df[[
        "feedback_text",
        "sentiment"
    ]].head(20)
)

                          feedback_text sentiment
0              Driver was rude and late  Negative
1        Excellent driving and behavior  Positive
2   Driver was very polite and friendly  Positive
3                    Average experience   Neutral
4                   Reached destination   Neutral
5               Amazing ride experience  Positive
6           Very happy with the service  Positive
7                    Average experience   Neutral
8         Terrible service and behavior  Negative
9           Very happy with the service  Positive
10                       Trip completed   Neutral
11          Very happy with the service  Positive
12      Very disappointed with the ride  Negative
13                      Reached on time   Neutral
14              Amazing ride experience  Positive
15               Driver arrived on time  Positive
16       Excellent driving and behavior  Positive
17      Ride was smooth and comfortable  Positive
18                      Reached on time   Neutral


In [ ]:
#NLP PIPELINE

In [33]:
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text

df["clean_feedback"] = df["feedback_text"].apply(
    clean_text
)

In [ ]:
#FEATURES & TARGET

In [34]:
X = df["clean_feedback"]

y = df["sentiment"]

In [ ]:
#TRAIN TEST SPLIT

In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

In [ ]:
#TF-IDF

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(

    max_features=5000,

    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

In [ ]:
#MODEL

In [37]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000
)

model.fit(
    X_train_tfidf,
    y_train
)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [ ]:
#PREDICTION

In [38]:
pred = model.predict(
    X_test_tfidf
)

In [ ]:
#EVALUATION

In [39]:
from sklearn.metrics import (

    accuracy_score,

    classification_report
)

print("Accuracy:")
print(
    accuracy_score(
        y_test,
        pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        pred
    )
)

Accuracy:
1.0

Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00      2301
     Neutral       1.00      1.00      1.00      3852
    Positive       1.00      1.00      1.00      3847

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



In [40]:
import joblib

joblib.dump(
    model,
    "sentiment_analysis_model.pkl"
)

joblib.dump(
    tfidf,
    "tfidf_vectorizer.pkl"
)

print("NLP Model Saved Successfully")

NLP Model Saved Successfully


In [41]:
import os

print(os.listdir())

['.anaconda', '.bash_history', '.cache', '.conda', '.continuum', '.git', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.keras', '.lesshst', '.matplotlib', '.ms-ad', '.streamlit', 'add', 'advanced_zends_50k_dataset.csv', 'Aerial_object_classification.ipynb', 'anaconda3', 'anaconda_projects', 'AppData', 'Application Data', 'Average_Youth_Illiteracy', 'Average_Youth_Illiteracy_Male_vs_Female.png', 'balanced_dataset.csv', 'basic 1.ipynb', 'best_model.pkl', 'Bottom 5 Countries by Average Youth Literacy.png', 'Bottom_5_Countries_by_Average_Youth_Literacy.png', 'catboost_info', 'cluster_encoder.pkl', 'cluster_scaler.pkl', 'collecting.ipynb', 'Contacts', 'Cookies', 'cover_type2.ipynb', 'cover_type_encoder.pkl', 'cover_type_project.ipynb', 'cross market guvi.ipynb', 'data collecting and merge.ipynb', 'deep learning.ipynb', 'Desktop', 'df_literacy_1990_2020.csv', 'df_literacy_1993_2023.csv', 'diamond_cluster_model.pkl', 'diamond_price_model.pkl', 'diamond_project.ipynb', 'Document